In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import linregress

In [ ]:
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')

In [ ]:
# Load the MS disease type data from the Excel file
ms_data = pd.read_excel(root / 'MSData_noNMO.xlsx')

# Remove any trailing whitespaces in the 'Disease Course/ Type at MRI date' column
ms_data['Disease Course/ Type at MRI date'] = ms_data['Disease Course/ Type at MRI date'].str.strip()

# Load choroid plexus volumes for healthy and pathologic subjects
volumes_healthy = pd.read_csv(root / 'Dataset002_DFGFinetuned_3d_fullres_full_new_hc___volumes.csv')
volumes_pathologic = pd.read_csv(root / 'Dataset002_DFGFinetuned_3d_fullres_full_ms___volumes.csv')

# Rename columns to ensure consistency for merging
ms_data = ms_data.rename(columns={'patID': 'subject_id', 'Disease Course/ Type at MRI date': 'MS_Disease_Type', 'DisDur_Months': 'Disease_Duration_Months'})

# Merge the pathologic volumes with MS disease type information
pathologic_data = pd.merge(volumes_pathologic, ms_data, on='subject_id', how='left')

# Add a column indicating health status
volumes_healthy['MS_Disease_Type'] = 'Healthy'
volumes_healthy['Disease_Duration_Months'] = 0  # Assume healthy subjects have 0 disease duration
pathologic_data['MS_Disease_Type'] = pathologic_data['MS_Disease_Type'].fillna('Unknown')
# Filter out entries with unknown disease type
pathologic_data = pathologic_data[pathologic_data['MS_Disease_Type'] != 'Unknown']

# Combine healthy and pathologic data
all_data = pd.concat([volumes_healthy, pathologic_data], ignore_index=True)

In [ ]:
# Calculate mean and standard deviation for each MS group
stats = all_data.groupby('MS_Disease_Type')['volume'].agg(['mean', 'std']).reset_index()

# Set the order of categories to ensure 'Healthy' comes first
stats['MS_Disease_Type'] = pd.Categorical(stats['MS_Disease_Type'], categories=['Healthy'] + [x for x in stats['MS_Disease_Type'].unique() if x != 'Healthy'], ordered=True)
stats = stats.sort_values('MS_Disease_Type')

# Assign colors to each MS disease type
unique_groups = all_data['MS_Disease_Type'].unique()
colors = plt.cm.tab10(range(len(unique_groups)))
color_map = dict(zip(unique_groups, colors))

# Plotting the results with different colors for each group
plt.figure(figsize=(10, 6))
plt.bar(stats['MS_Disease_Type'], stats['mean'], yerr=stats['std'], color=[color_map[group] for group in stats['MS_Disease_Type']], capsize=5)
plt.xlabel('MS Disease Type')
plt.ylabel('Choroid Plexus Volume (Mean ± STD)')
plt.title('Mean and STD of Choroid Plexus Volumes by MS Disease Type')
plt.xticks(rotation=45)
plt.show()
#plt.savefig('volume_dist_per_type.png')
#plt.close()

In [ ]:
# Plotting box plots for choroid plexus volume by MS Disease Type with custom colors and stripplot overlay
plt.figure(figsize=(12, 6))

# Plot each boxplot separately with its color
for i, group in enumerate(unique_groups):
    subset = all_data[all_data['MS_Disease_Type'] == group]
    sns.boxplot(data=subset, x='MS_Disease_Type', y='volume', color=color_map[group],
                showmeans=True, meanline=True,
                meanprops={"color": "blue", "ls": "--", "lw": 1.5},
                boxprops={'facecolor': color_map[group], 'alpha': 0.6},
                flierprops={'markerfacecolor': color_map[group], 'markeredgecolor': color_map[group]},
                whiskerprops={'color': color_map[group]},
                capprops={'color': color_map[group]})

# Overlay a strip plot for individual data points
sns.stripplot(data=all_data, x='MS_Disease_Type', y='volume', color='black', size=3, jitter=0.1, alpha=0.6)

plt.xlabel('MS Type')
plt.ylabel('Choroid Plexus Volume')
plt.title('Choroid Plexus Volumes by MS Type')
plt.xticks(rotation=45)
plt.show()
#plt.savefig('volume_dist_per_type.png')
#plt.close()

In [ ]:
# Scatter subplots with regression lines, excluding 'Healthy' group
unique_groups_no_healthy = [group for group in unique_groups if group != 'Healthy']
num_groups = len(unique_groups_no_healthy)
fig, axs = plt.subplots(nrows=1, ncols=num_groups, figsize=(num_groups * 5, 5), sharey=False)

for i, group in enumerate(unique_groups_no_healthy):
    subset = all_data[all_data['MS_Disease_Type'] == group]

    # Scatter plot
    axs[i].scatter(subset['Disease_Duration_Months'], subset['volume'], color=color_map[group], label=group)

    # Regression line
    slope, intercept, r, p, se = linregress(subset['Disease_Duration_Months'], subset['volume'])
    x_vals = np.array(axs[i].get_xlim())
    y_vals = intercept + slope * x_vals
    axs[i].plot(x_vals, y_vals, color=color_map[group], linestyle='--', label='Trend')

    axs[i].set_title(f'{group} p={p.round(4)}')
    axs[i].set_xlabel('Disease Duration (Months)')
    axs[i].legend()

# Set common y-axis label
fig.supylabel('Choroid Plexus Volume')
plt.suptitle('Choroid Plexus Volume vs Disease Duration by MS Type', y=1.05)
plt.tight_layout()
plt.show()
#plt.savefig('volume_per_month.png')
#plt.close()